## Housing Preprocessing

This notebook describes the preprocessing steps applied to the apartment sales data scraped from Booli.se.

The dataset contains the following features:

* **Selldate** – Date when the apartment was sold
* **Street** – Street address of the apartment
* **District** – District where the apartment is located
* **Municipality** – Municipality where the apartment is located
* **Selling price** – Final selling price of the apartment
* **Livingspace** – Living area in square meters
* **Secondaryspace** – Secondary area in square meters
* **Rooms** – Number of rooms
* **Floor** – Floor on which the apartment is located
* **Valuechange** – Change in value relative to the initial selling price
* **Priceperm2** – Selling price per square meter
* **Weblink** – Link to the corresponding Booli.se listing

The scraping tool does not always capture all feature values in their correct columns. In particular, missing values can cause subsequent values to shift into incorrect columns, resulting in an improperly structured dataset. The affected values were therefore manually reorganized into their correct columns using Excel.

The following sections describe the preprocessing steps applied to the CSV file generated from Excel.


In [262]:
from pathlib import Path

import pandas as pd


In [263]:
FILE_PATH = Path().cwd() / "sold_apartments.csv"

In [264]:
df = pd.read_csv(FILE_PATH, sep=";")
df

,Selldate,Street,District,Municipality,Sellingprice,Livingspace,Secondaryspace,Rooms,Floor,Valuechange,Priceperm2,Weblink
0,2026-01-30,Karlavägen73,Östermalm,Stockholm,47000000,284,NaN,8,3,"0,068",165500,https://www.booli.se/sok/slutpriser?areaIds=2&...
1,2026-04-21,Adresssaknas,Östermalm,Stockholm,37500000,246,NaN,8,5,"0,042",152400,https://www.booli.se/sok/slutpriser?areaIds=2&...
2,2026-04-17,Hantverkargatan79B,Kungsholmen,Stockholm,26800000,168,2,7,5,"0,081",159500,https://www.booli.se/sok/slutpriser?areaIds=2&...
3,2025-10-23,MagnusLadulåsgatan49,Maria,Stockholm,13900000,150,6,7,4,"0,163",92700,https://www.booli.se/sok/slutpriser?areaIds=2&...
4,2026-03-12,Karlavägen71,Östermalm,Stockholm,40800000,272,NaN,7,3,"-0,092",150000,https://www.booli.se/sok/slutpriser?areaIds=2&...
...,...,...,...,...,...,...,...,...,...,...,...,...
40491,2025-11-06,SicklaAllé65,Sickla,Nacka,3700000,51,NaN,NaN,NaN,NaN,72500,https://www.booli.se/sok/slutpriser?areaIds=2&...
40492,2025-12-06,Broderivägen23,Bromma,Stockholm,2500000,49,NaN,NaN,2,NaN,51000,https://www.booli.se/sok/slutpriser?areaIds=2&...
40493,2025-11-06,Spångavägen49,Bromma,Stockholm,2400000,"41,5",NaN,NaN,NaN,"0,046",57800,https://www.booli.se/sok/slutpriser?areaIds=2&...
40494,2026-06-08,Östermalmsgatan21,Östermalm,Stockholm,6600000,40,NaN,NaN,2,"0,016",165000,https://www.booli.se/sok/slutpriser?areaIds=2&...


* Check if string columns have any weird labels for missing values such as "Adresssaknas"
* Drop Weblink as they have no value

In [265]:
df.drop(columns=["Weblink"], inplace=True)

string_columns = ["Street", "District", "Municipality"]

for col in string_columns:
    sakna_count = df[col].str.contains(r'sakna', regex=True).sum()
    empty_count = df[col].str.contains(r'^\s*$', regex=True).sum() # ^\s*$: matches empty strings or strings with only
                                                                   # whitespace
    print(f"Column: {col}, Contains 'sakna' count: {sakna_count}, empty count: {empty_count}")


Column: Street, Contains 'sakna' count: 5, empty count: 0
Column: District, Contains 'sakna' count: 0, empty count: 0
Column: Municipality, Contains 'sakna' count: 0, empty count: 0


In [266]:
# Street contains five instances of "sakna". Replace with NaN
df["Street"] = df["Street"].replace(r'sakna', pd.NA, regex=True)

In [267]:
df

,Selldate,Street,District,Municipality,Sellingprice,Livingspace,Secondaryspace,Rooms,Floor,Valuechange,Priceperm2
0,2026-01-30,Karlavägen73,Östermalm,Stockholm,47000000,284,NaN,8,3,"0,068",165500
1,2026-04-21,NaN,Östermalm,Stockholm,37500000,246,NaN,8,5,"0,042",152400
2,2026-04-17,Hantverkargatan79B,Kungsholmen,Stockholm,26800000,168,2,7,5,"0,081",159500
3,2025-10-23,MagnusLadulåsgatan49,Maria,Stockholm,13900000,150,6,7,4,"0,163",92700
4,2026-03-12,Karlavägen71,Östermalm,Stockholm,40800000,272,NaN,7,3,"-0,092",150000
...,...,...,...,...,...,...,...,...,...,...,...
40491,2025-11-06,SicklaAllé65,Sickla,Nacka,3700000,51,NaN,NaN,NaN,NaN,72500
40492,2025-12-06,Broderivägen23,Bromma,Stockholm,2500000,49,NaN,NaN,2,NaN,51000
40493,2025-11-06,Spångavägen49,Bromma,Stockholm,2400000,"41,5",NaN,NaN,NaN,"0,046",57800
40494,2026-06-08,Östermalmsgatan21,Östermalm,Stockholm,6600000,40,NaN,NaN,2,"0,016",165000


In [268]:
df.dtypes

Selldate            str
Street              str
District            str
Municipality        str
Sellingprice      int64
Livingspace         str
Secondaryspace      str
Rooms               str
Floor               str
Valuechange         str
Priceperm2          str
dtype: object

* Convert numerical features from strings to floats.
* Convert date to date object.

In [269]:
numerical_features = ["Livingspace", "Secondaryspace", "Rooms", "Floor", "Valuechange", "Priceperm2"]

df["Selldate"] = pd.to_datetime(df["Selldate"])

# Sellingprice is converted to float as SEK is measured in kronor and ören.
df["Sellingprice"] = df["Sellingprice"].astype(float)

for feature in numerical_features:
    commas = df[feature].str.contains(',').sum()
    periods = df[feature].str.contains('\\.').sum()
    print(f"Feature: {feature}, commas: {commas}, periods: {periods}")


Feature: Livingspace, commas: 7110, periods: 0
Feature: Secondaryspace, commas: 24, periods: 0
Feature: Rooms, commas: 2927, periods: 0
Feature: Floor, commas: 1315, periods: 0
Feature: Valuechange, commas: 29804, periods: 0
Feature: Priceperm2, commas: 0, periods: 0


In [270]:
# Majority of features are numerical, so we replace commas with periods to standardize the decimal format.
df[numerical_features] = df[numerical_features].replace(',', '.', regex=True)

In [271]:
# We attempt to convert all numerical features to float and identify any problematic features
problematic_features = []
for feature in numerical_features:
    try:
        df[feature] = df[feature].astype(float)
    except ValueError:
        problematic_features.append(feature)

problematic_features

['Priceperm2']

In [272]:
problematic_values = {}
for feature in problematic_features:
    problematic_values[feature] = []
    for value in df[feature]:
        try:
            float(value)
        except ValueError:
            problematic_values[feature].append(value)

print("Problematic features: \n", problematic_values)

Problematic features: 
 {'Priceperm2': ['0tomt', '241tomt', '2097tomt', '2054tomt']}


In [273]:
# Replace problematic values with NaN so that the features can be converted to float
for feature in problematic_features:
    for value in problematic_values[feature]:
        df[feature] = df[feature].replace(value, pd.NA)

In [274]:
# Convert each problematic feature to float
for feature in problematic_features:
    df[feature] = df[feature].astype(float)

In [275]:
df.dtypes

Selldate          datetime64[us]
Street                       str
District                     str
Municipality                 str
Sellingprice             float64
Livingspace              float64
Secondaryspace           float64
Rooms                    float64
Floor                    float64
Valuechange              float64
Priceperm2               float64
dtype: object

* Investigate if Priceperm2 can be dropped as this value can be derived from Sellingprice and Livingspace

In [276]:
features = ["Sellingprice", "Livingspace", "Priceperm2"]

sell_live_price_df = df[features]
sell_live_price_df.isnull().sum()

Sellingprice     0
Livingspace     83
Priceperm2      87
dtype: int64

In [277]:
# Select the empty living space entries
empty_livingspace_df = sell_live_price_df[sell_live_price_df["Livingspace"].isnull()]
empty_livingspace_df
empty_livingspace_df.isnull().sum()

Sellingprice     0
Livingspace     83
Priceperm2      83
dtype: int64

In [278]:
# Pick the Priceperm2 entries that are NaN but has living space filled
empty_priceperm2_df = sell_live_price_df[
    sell_live_price_df["Priceperm2"].isnull() & sell_live_price_df["Livingspace"].notnull()
]
empty_priceperm2_df
empty_priceperm2_df.isnull().sum()

Sellingprice    0
Livingspace     0
Priceperm2      4
dtype: int64

It appears that 83 Priceperm2 values cannot be derived because the corresponding Livingspace values are missing. However, Priceperm2 can be calculated for the remaining observations using Selling price and Livingspace. Since Priceperm2 can therefore be reconstructed for all observations except the 83 rows with missing Livingspace, the Priceperm2 column can potentially be removed, assuming that Livingspace and Priceperm2 contain reliable values. To verify this, we calculate the price per square metre using Selling price and Livingspace and compare the resulting values with the existing Priceperm2 values.

In [279]:
sell_live_price_df["price_per_sqm"] = sell_live_price_df["Sellingprice"] / sell_live_price_df["Livingspace"]

sell_live_price_df["difference"] = abs(sell_live_price_df["price_per_sqm"] - sell_live_price_df["Priceperm2"])

In [280]:
sell_live_price_df["difference"].describe().T

count    40409.000000
mean        24.203082
std         15.093090
min          0.000000
25%         11.111111
50%         24.691358
75%         36.842105
max         50.000000
Name: difference, dtype: float64

The maximum difference is only 50 SEK, which is negligible compared with apartment selling prices, which are typically in the millions of SEK. Therefore, Priceperm2 can be considered redundant and removed from the dataset.

In [281]:
df.drop(columns=["Priceperm2"], inplace=True)


In [282]:
df.head(10)

,Selldate,Street,District,Municipality,Sellingprice,Livingspace,Secondaryspace,Rooms,Floor,Valuechange
0,2026-01-30,Karlavägen73,Östermalm,Stockholm,47000000.0,284.0,NaN,8.0,3.0,0.068
1,2026-04-21,NaN,Östermalm,Stockholm,37500000.0,246.0,NaN,8.0,5.0,0.042
2,2026-04-17,Hantverkargatan79B,Kungsholmen,Stockholm,26800000.0,168.0,2.0,7.0,5.0,0.081
3,2025-10-23,MagnusLadulåsgatan49,Maria,Stockholm,13900000.0,150.0,6.0,7.0,4.0,0.163
4,2026-03-12,Karlavägen71,Östermalm,Stockholm,40800000.0,272.0,NaN,7.0,3.0,-0.092
5,2025-09-17,Lill-JansPlan5,Östermalm,Stockholm,32000000.0,237.0,NaN,7.0,2.0,-0.045
6,2025-12-01,Valhallavägen129,Gärdet,Stockholm,27500000.0,193.0,NaN,7.0,NaN,0.000
7,2026-02-18,Grenljusbacken26,Gröndal,Stockholm,13900000.0,172.0,NaN,7.0,11.0,0.163
8,2026-01-14,Näsbydalsvägen8,Näsbydal,Täby,7250000.0,171.5,NaN,7.0,16.0,0.000
9,2026-06-13,Majorsvägen7A,Barkarbystaden,Järfälla,4500000.0,165.0,NaN,7.0,4.0,-0.020


In [283]:
df.to_csv("processed_housing_data.csv", index=False)